In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")


C:\Users\KRESHAN88\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer.eos_token

'<|im_end|>'

In [3]:
from transformers import AutoTokenizer
import torch

# Load the Qwen 2.5 7B Instruct tokenizer
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Set the padding token to EOS, as done previously
tokenizer.pad_token = tokenizer.eos_token

#               4= 1024 - 1020
# Padding Tokens=Length of Longest Sequence in Batch−Length of Current Sequence
sentences = [
    "Hello, how are you?",  # Shorter sentence
    "The sun is shining brightly today, which is lovely." # Longer sentence
]

def print_padding_example(padding_side, sentences):
    """Tokenizes, pads, and prints results for a given padding side."""
    tokenizer.padding_side = padding_side

    # Tokenize the batch and apply padding
    output = tokenizer(sentences, padding=True, return_tensors="pt")

    # Decode the sequences back to text to visualize the padding
    decoded_sequences = [tokenizer.decode(seq, skip_special_tokens=False) for seq in output['input_ids']]

    # Get the padding token string for display
    pad_token_str = tokenizer.decode(tokenizer.pad_token_id, skip_special_tokens=False).strip()

    print(f"\n--- {padding_side.upper()} PADDING ---")

    for i, seq_id in enumerate(output['input_ids']):
        print(f"\nOriginal Text {i}: '{sentences[i]}'")
        print(f"Token IDs   {i}: {seq_id.tolist()}")
        print(f"Attention M.{i}: {output['attention_mask'][i].tolist()}")

        # This shows where the PAD token is inserted in the decoded text
        print(f"Decoded Text {i}: '{decoded_sequences[i].replace(pad_token_str, f'|{pad_token_str}|')}'")
        print(f"Padding Token: {pad_token_str} (ID: {tokenizer.pad_token_id})")

# Print the examples for both modes
print_padding_example("right", sentences)
print_padding_example("left", sentences)


--- RIGHT PADDING ---

Original Text 0: 'Hello, how are you?'
Token IDs   0: [9707, 11, 1246, 525, 498, 30, 151645, 151645, 151645, 151645, 151645]
Attention M.0: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
Decoded Text 0: 'Hello, how are you?|<|im_end|>||<|im_end|>||<|im_end|>||<|im_end|>||<|im_end|>|'
Padding Token: <|im_end|> (ID: 151645)

Original Text 1: 'The sun is shining brightly today, which is lovely.'
Token IDs   1: [785, 7015, 374, 47925, 75289, 3351, 11, 892, 374, 16690, 13]
Attention M.1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Decoded Text 1: 'The sun is shining brightly today, which is lovely.'
Padding Token: <|im_end|> (ID: 151645)

--- LEFT PADDING ---

Original Text 0: 'Hello, how are you?'
Token IDs   0: [151645, 151645, 151645, 151645, 151645, 9707, 11, 1246, 525, 498, 30]
Attention M.0: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
Decoded Text 0: '|<|im_end|>||<|im_end|>||<|im_end|>||<|im_end|>||<|im_end|>|Hello, how are you?'
Padding Token: <|im_end|> (ID: 151645)

Original Text 1: 'The s

In [4]:

def format_dolly(example):


    system_content = example["instruction"]

    user_content = example["context"] if example["context"] else ""

    # Use response as the assistant message
    assistant_content = example["response"]

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]

    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    return {"text": formatted_text}

processed_dataset = dataset.map(format_dolly, remove_columns=dataset.column_names)


In [5]:
processed_dataset

Dataset({
    features: ['text'],
    num_rows: 15011
})

In [6]:

def tokenize_function(examples):
    # Tokenize the 'text' column, which now holds the chat template formatted string
    return tokenizer(examples["text"], truncation=True, max_length=1024)

tokenized_dataset = processed_dataset.map(tokenize_function, batched=True)



In [7]:
final_training_column = tokenized_dataset.remove_columns(["text", "attention_mask"])
print(final_training_column[0]["input_ids"])

[151644, 8948, 198, 4498, 1521, 11214, 8330, 1191, 10350, 30, 151645, 198, 151644, 872, 198, 63697, 8330, 11, 279, 11133, 829, 315, 11214, 8330, 34130, 78992, 12324, 11, 374, 458, 13369, 5980, 32475, 13, 1084, 374, 279, 7772, 32475, 553, 25099, 1379, 311, 990, 279, 11214, 6741, 13, 1084, 64262, 3516, 389, 220, 18, 16, 6156, 220, 17, 15, 15, 15, 438, 11214, 8697, 11, 448, 1378, 14135, 389, 264, 3175, 6021, 13, 1084, 14843, 1730, 5086, 438, 264, 3598, 32475, 304, 8330, 594, 12728, 3081, 1283, 279, 18179, 315, 1527, 66514, 8330, 304, 6122, 220, 17, 15, 15, 16, 13, 576, 32475, 702, 2474, 14700, 311, 5961, 8683, 220, 18, 17, 9720, 304, 8330, 11, 504, 68676, 304, 46235, 11, 26437, 323, 21273, 13, 151645, 198, 151644, 77091, 198, 63697, 8330, 64262, 3516, 389, 220, 18, 16, 6156, 220, 17, 15, 15, 15, 438, 11214, 8697, 11, 448, 1378, 14135, 389, 264, 3175, 6021, 13, 151645, 198]


In [8]:
final_training_column = tokenized_dataset.remove_columns(["text", "attention_mask"])
print("First 10 rows of the final 'input_ids' column:")
for i in range(10):
    print(final_training_column[i]["input_ids"])

First 10 rows of the final 'input_ids' column:
[151644, 8948, 198, 4498, 1521, 11214, 8330, 1191, 10350, 30, 151645, 198, 151644, 872, 198, 63697, 8330, 11, 279, 11133, 829, 315, 11214, 8330, 34130, 78992, 12324, 11, 374, 458, 13369, 5980, 32475, 13, 1084, 374, 279, 7772, 32475, 553, 25099, 1379, 311, 990, 279, 11214, 6741, 13, 1084, 64262, 3516, 389, 220, 18, 16, 6156, 220, 17, 15, 15, 15, 438, 11214, 8697, 11, 448, 1378, 14135, 389, 264, 3175, 6021, 13, 1084, 14843, 1730, 5086, 438, 264, 3598, 32475, 304, 8330, 594, 12728, 3081, 1283, 279, 18179, 315, 1527, 66514, 8330, 304, 6122, 220, 17, 15, 15, 16, 13, 576, 32475, 702, 2474, 14700, 311, 5961, 8683, 220, 18, 17, 9720, 304, 8330, 11, 504, 68676, 304, 46235, 11, 26437, 323, 21273, 13, 151645, 198, 151644, 77091, 198, 63697, 8330, 64262, 3516, 389, 220, 18, 16, 6156, 220, 17, 15, 15, 15, 438, 11214, 8697, 11, 448, 1378, 14135, 389, 264, 3175, 6021, 13, 151645, 198]
[151644, 8948, 198, 23085, 374, 264, 9419, 315, 7640, 30, 2014, 375, 4

In [9]:

# 1. Original Dataset Row
example = dataset[0]
print("--- Original Data (Example 0) ---")
print(f"Instruction (System): {example['instruction']}")
print(f"Context (User): {example['context']}")
print(f"Response (Assistant): {example['response']}")
print("-" * 40)

# 2. Chat Templated String (Re-run for display)
system_content = example["instruction"]
user_content = example["context"] if example["context"] else "None."
assistant_content = example["response"]

messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": user_content},
    {"role": "assistant", "content": assistant_content},
]
formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

print("--- Chat Templated String ('text' column) ---")
print(formatted_text)
print("-" * 40)

# 3. Tokenized Output (Input IDs and Attention Mask)
input_ids = tokenized_dataset[0]["input_ids"]
attention_mask = tokenized_dataset[0]["attention_mask"] # This is where the mask is pulled from the tokenized_dataset

decoded_tokens = tokenizer.decode(input_ids)

print("--- Tokenized Output (Input IDs) ---")
print(input_ids)
print(f"Length of Input IDs: {len(input_ids)}")
print("-" * 40)

# Display the Attention Mask
print("--- Attention Mask ---")
print(attention_mask)
print(f"Length of Attention Mask: {len(attention_mask)}")
print("-" * 40)

print("--- Decoded Token String (for verification) ---")
print(decoded_tokens)

--- Original Data (Example 0) ---
Instruction (System): When did Virgin Australia start operating?
Context (User): Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.
Response (Assistant): Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.
----------------------------------------
--- Chat Templated String ('text' column) ---
<|im_start|>system
When did Virgin Australia start operating?<|im_end|>
<|im_start|>user
Virgin Australia, the trading name of Virgin Australia Airlines Pty L